In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(r"C:\mle-01-p1-team3")
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from retrieve import retrieve

C:\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3055.58it/s]

### Retriever 반환 형태 확인

In [2]:
question = "표창을 사용하는 직업 모두 정리해봐"

retrieved_docs = retrieve(question, k=5)

print("반환 타입:", type(retrieved_docs))
print("검색 결과 수:", len(retrieved_docs))

if retrieved_docs:
    print("첫 번째 결과 타입:", type(retrieved_docs[0]))
    print("첫 번째 결과:")
    print(retrieved_docs[0])
else:
    print("검색 결과가 없습니다.")

검색 범위: jobs


반환 타입: <class 'list'>
검색 결과 수: 5
첫 번째 결과 타입: <class 'dict'>
첫 번째 결과:
{'rank': 1, 'id': 'chunk_1472', 'page_content': '직업명: 나이트로드\n별명: 그림자 속에 숨은 존재\n설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.\n주스탯: LUK (행운)\n무기: 표창, 아대', 'metadata': {'category': '도적', 'chunk_index': 1472, 'url': 'https://maplestory.nexon.com/Guide/N23Job/View/30', 'source': 'jobs', 'name': '나이트로드', 'job_id': '30'}, 'distance': 0.5280954539775848, 'score': 0.47190454602241516}


In [3]:
for doc in retrieved_docs:
    metadata = doc.get("metadata") or {}
    print(
        metadata.get("source"),
        metadata.get("name"),
        f"score={doc.get('score', 0):.4f}",
    )

jobs 나이트로드 score=0.4719
jobs 보우마스터 score=0.4298
jobs 배틀메이지 score=0.3988
jobs 나이트워커 score=0.3951
jobs 아크메이지(썬,콜) score=0.3628


### Retriever 결과를 LLM context 문자열로 변환

In [4]:
# build_context()는 src/build_context.py로 옮겼습니다.
# 노트북 셀에 두면 복붙으로 퍼지면서 팀원마다 버전이 달라집니다.
from build_context import build_context, has_brace

print(build_context.__doc__)


    Retriever 결과(list[dict])를 LLM에 전달할 context 문자열로 변환합니다.

    documents: retrieve()가 돌려주는 리스트. 각 항목은
        {"id", "page_content", "metadata", "rank", "score", "distance"} 형태.

    검색 결과가 없으면 NO_RESULT_MESSAGE를 돌려줍니다. 빈 문자열을 돌려주면
    프롬프트의 [Context] 항목이 통째로 비어 LLM이 자유롭게 지어내기 쉬워집니다.
    


In [5]:
question = "표창을 사용하는 직업 모두 정리해봐"

retrieved_docs = retrieve(question, k=5)
context = build_context(retrieved_docs)

print(context)

검색 범위: jobs
[검색 결과 1]
문서 유형: jobs
문서 제목: 나이트로드
chunk_id: chunk_1472

내용:
직업명: 나이트로드
별명: 그림자 속에 숨은 존재
설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.
주스탯: LUK (행운)
무기: 표창, 아대

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/30

---

[검색 결과 2]
문서 유형: jobs
문서 제목: 보우마스터
chunk_id: chunk_1465

내용:
직업명: 보우마스터
별명: 속사의 정점
설명: 활의 정점에 도달해 다양한 화살로 적을 섬멸하는 궁수입니다. 화살을 연속적으로 발사하는 속사 공격과 상황에 맞춰 기능을 선택할 수 있는 추가 화살을 끊임없이 퍼부어 전장을 뒤덮습니다.
주스탯: DEX (민첩)
무기: 활

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/23

---

[검색 결과 3]
문서 유형: jobs
문서 제목: 배틀메이지
chunk_id: chunk_1458

내용:
직업명: 배틀메이지
별명: 최전선의 마법사
설명: 실전에 특화되어 근접전이 가능한 마법사입니다. 높은 기동력으로 접근해 스태프를 휘둘러 공격하며, 어둠의 힘으로 적을 응징하고 다양한 오라로 동료를 지원합니다.
주스탯: INT (지력)
무기: 스태프

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/17

---

[검색 결과 4]
문서 유형: jobs
문서 제목: 나이트워커
chunk_id: chunk_1475

내용:
직업명: 나이트워커
별명: 비정한 어둠의 기사
설명: 정령 다크니스의 힘을 받아들여 그림자와 어둠의 힘을 사용하는 도적입니다. 그림자로 빚어낸 배트와 자신의 행동을 따라하는 그림

### build_context() 엣지 케이스

정상 경로만 확인하면 팀원 쪽에서 터집니다. 검색 결과가 없거나 metadata가 빠진 경우를 함께 확인합니다.

In [6]:
# 검색 결과 없음 — 빈 문자열이 아니라 안내 문구를 돌려줘야 합니다.
# 빈 문자열이면 프롬프트의 [Context]가 통째로 비어 LLM이 지어내기 쉬워집니다.
print("빈 리스트:", repr(build_context([])))
print("None    :", repr(build_context(None)))

# metadata가 없거나 비어 있는 문서
broken_docs = [
    {"id": "chunk_x", "page_content": "본문만 있는 문서"},
    {"id": "chunk_y", "page_content": "메타데이터가 None", "metadata": None},
    {"id": "chunk_z", "page_content": "name/title 없음", "metadata": {"source": "guide"}},
]

print()
print(build_context(broken_docs))

빈 리스트: '관련 검색 결과가 없습니다.'
None    : '관련 검색 결과가 없습니다.'

[검색 결과 1]
문서 유형: unknown
문서 제목: 제목 없음
chunk_id: chunk_x

내용:
본문만 있는 문서

출처 URL:

---

[검색 결과 2]
문서 유형: unknown
문서 제목: 제목 없음
chunk_id: chunk_y

내용:
메타데이터가 None

출처 URL:

---

[검색 결과 3]
문서 유형: guide
문서 제목: 제목 없음
chunk_id: chunk_z

내용:
name/title 없음

출처 URL:


### context가 prompt에 정상 삽입되는지 확인

`prompt.format_messages()`는 OpenAI 키 없이 동작합니다. 실제 호출 전에 여기서 삽입을 검증합니다.

주의할 점 하나 — `ChatPromptTemplate`은 중괄호 `{}`를 템플릿 변수로 해석합니다. context 안에 중괄호가 섞이면 `KeyError`가 납니다.

In [7]:
from langchain_core.prompts import ChatPromptTemplate

# rag_chain.ipynb과 동일한 프롬프트
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
당신은 메이플스토리 정보 안내 챗봇입니다.

반드시 제공된 Context를 기반으로 답변하세요.
Context에 없는 정보는 임의로 만들어내지 마세요.
""".strip(),
        ),
        (
            "human",
            """
[Context]
{context}

[Question]
{question}
""".strip(),
        ),
    ]
)

test_questions = [
    "표창을 사용하는 직업 모두 정리해봐",       # jobs 필터
    "큐브 잠재능력 등급 상승 확률 알려줘",       # items 필터
    "메이플 초보자에게 좋은 사냥터 추천해줘",     # 필터 없음(전체 검색)
]

for test_question in test_questions:
    context = build_context(retrieve(test_question, k=3))
    messages = prompt.format_messages(context=context, question=test_question)

    human_message = messages[1].content

    print(f"질문: {test_question}")
    print(f"  중괄호 포함: {has_brace(context)}")
    print(f"  context 삽입됨: {context in human_message}")
    print(f"  question 삽입됨: {test_question in human_message}")
    print(f"  '{{context}}' 치환 완료: {'{context}' not in human_message}")
    print(f"  human 메시지 길이: {len(human_message)}자")
    print()

검색 범위: jobs
질문: 표창을 사용하는 직업 모두 정리해봐
  중괄호 포함: False
  context 삽입됨: True
  question 삽입됨: True
  '{context}' 치환 완료: True
  human 메시지 길이: 863자

검색 범위: items


질문: 큐브 잠재능력 등급 상승 확률 알려줘
  중괄호 포함: False
  context 삽입됨: True
  question 삽입됨: True
  '{context}' 치환 완료: True
  human 메시지 길이: 669자

검색 범위: 전체 문서
질문: 메이플 초보자에게 좋은 사냥터 추천해줘
  중괄호 포함: False
  context 삽입됨: True
  question 삽입됨: True
  '{context}' 치환 완료: True
  human 메시지 길이: 1023자



In [8]:
# 실제로 LLM에 넘어가는 메시지를 눈으로 확인합니다.
question = "표창을 사용하는 직업 모두 정리해봐"
context = build_context(retrieve(question, k=3))

for message in prompt.format_messages(context=context, question=question):
    print(f"===== {message.type} =====")
    print(message.content)
    print()

검색 범위: jobs
===== system =====
당신은 메이플스토리 정보 안내 챗봇입니다.

반드시 제공된 Context를 기반으로 답변하세요.
Context에 없는 정보는 임의로 만들어내지 마세요.

===== human =====
[Context]
[검색 결과 1]
문서 유형: jobs
문서 제목: 나이트로드
chunk_id: chunk_1472

내용:
직업명: 나이트로드
별명: 그림자 속에 숨은 존재
설명: 다양한 종류의 표창을 능수능란하게 다루는 도적입니다. 끊임없는 표창 투척과 함께 암살자의 표식, 부적, 두루마리로 더욱 많은 표창을 소환하여 적에게 강력한 피해를 주고 전장을 제압합니다.
주스탯: LUK (행운)
무기: 표창, 아대

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/30

---

[검색 결과 2]
문서 유형: jobs
문서 제목: 보우마스터
chunk_id: chunk_1465

내용:
직업명: 보우마스터
별명: 속사의 정점
설명: 활의 정점에 도달해 다양한 화살로 적을 섬멸하는 궁수입니다. 화살을 연속적으로 발사하는 속사 공격과 상황에 맞춰 기능을 선택할 수 있는 추가 화살을 끊임없이 퍼부어 전장을 뒤덮습니다.
주스탯: DEX (민첩)
무기: 활

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/23

---

[검색 결과 3]
문서 유형: jobs
문서 제목: 배틀메이지
chunk_id: chunk_1458

내용:
직업명: 배틀메이지
별명: 최전선의 마법사
설명: 실전에 특화되어 근접전이 가능한 마법사입니다. 높은 기동력으로 접근해 스태프를 휘둘러 공격하며, 어둠의 힘으로 적을 응징하고 다양한 오라로 동료를 지원합니다.
주스탯: INT (지력)
무기: 스태프

출처 URL:
https://maplestory.nexon.com/Guide/N23Job/View/17

[Question]
표창을 사용하는 직업 모두 정리해봐



In [9]:
# 중괄호는 지금 검색되는 문서에만 없으면 되는 게 아니라, 코퍼스 전체에 없어야 안전합니다.
# 하나라도 걸리면 그 문서가 검색되는 순간 KeyError로 터집니다.
import chromadb

collection = chromadb.PersistentClient(path=str(PROJECT_ROOT / "chroma_db")).get_collection(
    "maplestory_guides"
)
all_documents = collection.get(include=["documents"])

brace_ids = [
    document_id
    for document_id, document in zip(all_documents["ids"], all_documents["documents"])
    if has_brace(document)
]

print(f"중괄호 포함 문서: {len(brace_ids)} / {len(all_documents['documents'])}")

if brace_ids:
    print("주의 — 아래 문서가 검색되면 prompt.format_messages()가 KeyError로 실패합니다.")
    print(brace_ids[:10])

중괄호 포함 문서: 0 / 3694


### 답변 추출 확인

`FakeListChatModel`로 실제 OpenAI 호출 없이 체인 배선을 검증합니다.
키가 없어도 돌아가고, 나중에 `ChatOpenAI`로 바꾸면 그대로 동작합니다.

여기서 확인하는 것은 **답변 내용이 아니라 배선**입니다.

- `retrieve → build_context → prompt → model → StrOutputParser`가 끊기지 않는지
- `StrOutputParser`가 `AIMessage`에서 `str`을 뽑아내는지
- 체인에 넣은 질문이 실제로 retriever까지 전달되는지

In [10]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

fake_model = FakeListChatModel(responses=["가짜 응답입니다."])

# 체인에 들어온 질문을 그대로 넘겨야 합니다.
# rag_chain.ipynb에는 lambda q: retrieve(question) 으로 되어 있는데,
# 인자 q를 버리고 전역변수 question을 쓰기 때문에 항상 같은 질문만 검색됩니다.
retrieve_runnable = RunnableLambda(lambda q: retrieve(q, k=3))

rag_chain = (
    {
        "context": retrieve_runnable | RunnableLambda(build_context),
        "question": RunnablePassthrough(),
    }
    | prompt
    | fake_model
    | StrOutputParser()
)

answer = rag_chain.invoke("표창을 사용하는 직업 모두 정리해봐")

# langchain_core 1.6.0의 StrOutputParser는 str이 아니라 TextAccessor를 돌려줍니다.
# str 서브클래스라 슬라이싱, ==, json.dumps, Streamlit 출력 모두 정상 동작합니다.
# 다만 type(answer) is str 로 검사하면 False이니 isinstance를 쓰세요.
print("답변 타입:", type(answer).__name__)
print("str 서브클래스:", isinstance(answer, str))
print("답변:", repr(str(answer)))

assert isinstance(answer, str), "StrOutputParser가 문자열을 돌려주지 않았습니다."

검색 범위: jobs


답변 타입: TextAccessor
str 서브클래스: True
답변: '가짜 응답입니다.'


In [11]:
# 체인에 넣은 질문이 실제로 retriever까지 도달하는지 확인합니다.
# retrieve를 감싸서 어떤 질문이 들어왔는지 기록합니다.
received_questions = []


def traced_retrieve(q):
    received_questions.append(q)
    return retrieve(q, k=3)


traced_chain = (
    {
        "context": RunnableLambda(traced_retrieve) | RunnableLambda(build_context),
        "question": RunnablePassthrough(),
    }
    | prompt
    | fake_model
    | StrOutputParser()
)

for test_question in test_questions:
    traced_chain.invoke(test_question)

print("체인에 넣은 질문 :", test_questions)
print("retriever가 받은 질문:", received_questions)
assert received_questions == test_questions, "질문이 retriever까지 전달되지 않았습니다."
print("\n배선 정상 — 이제 fake_model을 ChatOpenAI로 바꾸면 실제 답변이 나옵니다.")

검색 범위: jobs


검색 범위: items


검색 범위: 전체 문서
체인에 넣은 질문 : ['표창을 사용하는 직업 모두 정리해봐', '큐브 잠재능력 등급 상승 확률 알려줘', '메이플 초보자에게 좋은 사냥터 추천해줘']
retriever가 받은 질문: ['표창을 사용하는 직업 모두 정리해봐', '큐브 잠재능력 등급 상승 확률 알려줘', '메이플 초보자에게 좋은 사냥터 추천해줘']

배선 정상 — 이제 fake_model을 ChatOpenAI로 바꾸면 실제 답변이 나옵니다.
